# 🔀 02 — Backbone ViT + Ablation Study del Router
**Tasks 2.4, 2.5, 2.6, 2.7, 2.8, 2.9 — EPIC 2**

| Task | Descripción | Depende de |
|------|-------------|------------|
| 2.4 | Extraer CLS tokens de todos los datasets | Imágenes en disco |
| 2.5 | Router A — ViT + Linear + Softmax | Z_train, Z_val |
| 2.6 | Router B — GMM (5 componentes) | Z_train, Z_val |
| 2.7 | Router C — Naive Bayes | Z_train, Z_val |
| 2.8 | Router D — FAISS k-NN + PCA | Z_train, Z_val |
| 2.9 | Tabla comparativa del Ablation Study | Los 4 routers |

**Orden:** Correr celdas de Setup → Extracción CLS → Los 4 routers (en cualquier orden) → Ablation Study

## 0. Setup

In [2]:
import sys, torch, time, json, random
import numpy as np
from pathlib import Path

sys.path.insert(0, '/workspace/moe_medical_vision/src')

# ─── Paths ───────────────────────────────────────────────────────────────────
DATASET_PATHS = {
    'nih_chestxray':  '/workspace/moe_medical_vision/data/raw/nih',
    'isic2019':       '/workspace/moe_medical_vision/data/raw/isic',
    'osteoarthritis': '/workspace/moe_medical_vision/data/raw/osteoporosis/KLGrade/KLGrade',
    'luna16':         '/workspace/moe_medical_vision/data/raw/luna16',
    'pancreatic':     '/workspace/moe_medical_vision/data/raw/pancreatic',
}
CHECKPOINT_DIR  = Path('/workspace/moe_medical_vision/checkpoints')
EMBEDDINGS_DIR  = Path('/workspace/moe_medical_vision/embeddings')
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE  = 64   # más grande para extracción — solo forward pass, sin gradientes
NUM_WORKERS = 4
SEED        = 42

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'Embeddings → {EMBEDDINGS_DIR}')
print()

# Verificar checkpoints disponibles
print('Checkpoints disponibles:')
for f in sorted(CHECKPOINT_DIR.glob('expert*.pth')):
    ck = torch.load(f, map_location='cpu', weights_only=False)
    print(f'  ✅ {f.name} — F1: {ck.get("best_f1", 0):.4f}')

# Verificar embeddings ya extraídos
print()
print('Embeddings ya extraídos:')
for f in sorted(EMBEDDINGS_DIR.glob('*.npy')):
    shape = np.load(f, mmap_mode='r').shape
    print(f'  ✅ {f.name} — shape: {shape}')

Device: cuda
GPU: NVIDIA GeForce RTX 4090
Embeddings → /workspace/moe_medical_vision/embeddings

Checkpoints disponibles:
  ✅ expert1_nih_bce_nosampler_a.pth — F1: 0.2817
  ✅ expert1_nih_best.pth — F1: 0.3266
  ✅ expert1_nih_dbloss_best.pth — F1: 0.1798
  ✅ expert1_nih_enriched_best.pth — F1: 0.3279
  ✅ expert1_nih_final_best.pth — F1: 0.2660
  ✅ expert1_nih_improved_best.pth — F1: 0.3278
  ✅ expert1_nih_sweep_01.pth — F1: 0.3255
  ✅ expert1_nih_sweep_03.pth — F1: 0.2804
  ✅ expert1_nih_tierF_best.pth — F1: 0.5338
  ✅ expert2_isic_best.pth — F1: 0.7921
  ✅ expert3_oa_best.pth — F1: 0.8357
  ✅ expert4_luna16_MIP_best.pth — F1: 0.0000
  ✅ expert4_luna_fixed.pth — F1: 0.0000
  ✅ expert4_luna_sweep_01.pth — F1: 0.0000
  ✅ expert5_pancreatic_FAST_best.pth — F1: 0.0000

Embeddings ya extraídos:
  ✅ Z_train_isic2019.npy — shape: (21504, 192)
  ✅ Z_train_luna16.npy — shape: (711, 192)
  ✅ Z_train_nih_chestxray.npy — shape: (95296, 192)
  ✅ Z_train_osteoarthritis.npy — shape: (3776, 192)
  ✅ Z_

## 1. Instalar dependencias faltantes

In [3]:
!pip install faiss-cpu scikit-learn timm --quiet

---
## 2. Backbone ViT congelado — Task 2.4

Instanciamos ViT-Tiny preentrenado, congelamos todos sus pesos,
y creamos la función que extrae únicamente el token `[CLS]`.

Este backbone **no se entrena** — solo se usa para generar embeddings.

In [4]:
import timm
import torch.nn as nn
from torch.amp import autocast

# ─── ViT-Tiny preentrenado ────────────────────────────────────────────────────
VIT_MODEL = 'vit_tiny_patch16_224'   # d_model = 192

backbone = timm.create_model(VIT_MODEL, pretrained=True)

# Congelar TODOS los pesos — el backbone no se entrena
for param in backbone.parameters():
    param.requires_grad = False

backbone = backbone.to(DEVICE)
backbone.eval()

d_model = backbone.embed_dim  # 192 para ViT-Tiny
n_params = sum(p.numel() for p in backbone.parameters()) / 1e6

print(f'Backbone: {VIT_MODEL}')
print(f'  d_model (dim del CLS token): {d_model}')
print(f'  Parámetros: {n_params:.1f}M (todos congelados)')

# Test rápido — verificar que el CLS token tiene la shape correcta
with torch.no_grad():
    dummy = torch.rand(2, 3, 224, 224).to(DEVICE)
    features = backbone.forward_features(dummy)   # (B, N_tokens, d_model)
    cls_token = features[:, 0, :]                  # (B, d_model) — token [CLS]
    print(f'  Test CLS token shape: {cls_token.shape}  ✅')

Backbone: vit_tiny_patch16_224
  d_model (dim del CLS token): 192
  Parámetros: 5.7M (todos congelados)
  Test CLS token shape: torch.Size([2, 192])  ✅


## 3. Extracción masiva de CLS tokens — Task 2.4

Pasamos todos los datasets por el ViT congelado y guardamos los tensores en disco.
Esto se hace **una sola vez** — ahorra horas de cómputo en el ablation study.

Resultado por dataset:
- `Z_train_{dataset}.npy` → shape `(N_train, 192)`
- `Z_val_{dataset}.npy`   → shape `(N_val, 192)`
- `y_train_{dataset}.npy` → etiquetas de experto (0-4)
- `y_val_{dataset}.npy`   → etiquetas de experto (0-4)

In [5]:
from tqdm.notebook import tqdm
from data.datasets import get_dataloader
from data.adaptive_preprocessor import AdaptivePreprocessor

preprocessor = AdaptivePreprocessor().to(DEVICE)

def extract_cls_tokens(dataset_name, root, split, backbone, preprocessor):
    """
    Pasa un dataset completo por el preprocessor + backbone ViT
    y devuelve los CLS tokens con sus etiquetas de experto.
    """
    # Batch pequeño para 3D, grande para 2D
    bs = 1 if dataset_name in ('luna16', 'pancreatic') else BATCH_SIZE
    loader, ds = get_dataloader(
        dataset_name, root, split,
        batch_size=bs, 
        num_workers=0 if dataset_name in ('luna16', 'pancreatic') else NUM_WORKERS
    )

    all_cls, all_expert_ids = [], []

    backbone.eval()
    with torch.no_grad():
        pbar = tqdm(loader, desc=f'  [{dataset_name}] {split}', leave=False)
        for batch in pbar:
            images = batch['image'].to(DEVICE)
            expert_ids = batch['expert_id']

            # Preprocessor adaptativo — resize 2D o 3D automáticamente
            with autocast('cuda'):
                images = preprocessor(images)

                # Para 3D: tomar el slice central como proxy 2D para el ViT
                if len(images.shape) == 5:  # (B, C, D, H, W)
                    D = images.shape[2]
                    images = images[:, :, D // 2, :, :]  # slice central → (B, C, H, W)
                    if images.shape[1] == 1:
                        images = images.repeat(1, 3, 1, 1)
                # Resize a 224×224 para el ViT
                images = torch.nn.functional.interpolate(
                    images, size=(224, 224), mode='bilinear', align_corners=False
                )

                features  = backbone.forward_features(images)  # (B, N, d_model)
                cls_tokens = features[:, 0, :]                  # (B, d_model)

            all_cls.append(cls_tokens.cpu().float().numpy())
            all_expert_ids.append(expert_ids.numpy())

    Z = np.concatenate(all_cls)         # (N, 192)
    y = np.concatenate(all_expert_ids)  # (N,)
    return Z, y


# ─── Extraer para los datasets disponibles ───────────────────────────────────
DATASETS_TO_EXTRACT = ['nih_chestxray', 'isic2019', 'osteoarthritis']

DATASETS_TO_EXTRACT += ['luna16']

for ds_name in DATASETS_TO_EXTRACT:
    root = DATASET_PATHS[ds_name]
    if not Path(root).exists():
        print(f'⚠️  {ds_name}: ruta no encontrada, saltando')
        continue

    # Saltar si ya existe
    z_train_path = EMBEDDINGS_DIR / f'Z_train_{ds_name}.npy'
    if z_train_path.exists():
        shape = np.load(z_train_path, mmap_mode='r').shape
        print(f'⏭️  {ds_name}: ya extraído {shape}, saltando')
        continue

    print(f'\n📊 Extrayendo: {ds_name}')
    t0 = time.time()

    Z_train, y_train = extract_cls_tokens(ds_name, root, 'train', backbone, preprocessor)
    Z_val,   y_val   = extract_cls_tokens(ds_name, root, 'val',   backbone, preprocessor)

    np.save(EMBEDDINGS_DIR / f'Z_train_{ds_name}.npy', Z_train)
    np.save(EMBEDDINGS_DIR / f'Z_val_{ds_name}.npy',   Z_val)
    np.save(EMBEDDINGS_DIR / f'y_train_{ds_name}.npy', y_train)
    np.save(EMBEDDINGS_DIR / f'y_val_{ds_name}.npy',   y_val)

    elapsed = time.time() - t0
    print(f'  ✅ Z_train: {Z_train.shape} | Z_val: {Z_val.shape} | {elapsed:.0f}s')

print('\n✅ Extracción completa')

⏭️  nih_chestxray: ya extraído (95296, 192), saltando
⏭️  isic2019: ya extraído (21504, 192), saltando
⏭️  osteoarthritis: ya extraído (3776, 192), saltando
⏭️  luna16: ya extraído (711, 192), saltando

✅ Extracción completa


## 4. Cargar todos los embeddings combinados

Concatenamos los embeddings de todos los datasets en arrays únicos
`Z_train`, `Z_val`, `y_train`, `y_val` para entrenar los routers.

In [6]:
Z_train_list, Z_val_list = [], []
y_train_list, y_val_list = [], []

print('Cargando embeddings:')
for ds_name in DATASETS_TO_EXTRACT:
    z_tr = EMBEDDINGS_DIR / f'Z_train_{ds_name}.npy'
    z_va = EMBEDDINGS_DIR / f'Z_val_{ds_name}.npy'
    y_tr = EMBEDDINGS_DIR / f'y_train_{ds_name}.npy'
    y_va = EMBEDDINGS_DIR / f'y_val_{ds_name}.npy'

    if not z_tr.exists():
        print(f'  ⚠️  {ds_name}: no encontrado, saltando')
        continue

    Z_train_list.append(np.load(z_tr))
    Z_val_list.append(np.load(z_va))
    y_train_list.append(np.load(y_tr))
    y_val_list.append(np.load(y_va))
    print(f'  ✅ {ds_name}: train={np.load(z_tr).shape[0]} | val={np.load(z_va).shape[0]}')

Z_train = np.concatenate(Z_train_list).astype(np.float32)  # (N_train, 192)
Z_val   = np.concatenate(Z_val_list).astype(np.float32)    # (N_val, 192)
y_train = np.concatenate(y_train_list)                      # (N_train,) — expert_id
y_val   = np.concatenate(y_val_list)                        # (N_val,)

print(f'\nZ_train combinado: {Z_train.shape}')
print(f'Z_val   combinado: {Z_val.shape}')
print(f'Clases en y_train: {np.unique(y_train, return_counts=True)}')

Cargando embeddings:
  ✅ nih_chestxray: train=95296 | val=16818
  ✅ isic2019: train=21504 | val=3799
  ✅ osteoarthritis: train=3776 | val=953
  ✅ luna16: train=711 | val=177

Z_train combinado: (121287, 192)
Z_val   combinado: (21747, 192)
Clases en y_train: (array([0, 1, 2, 3]), array([95296, 21504,  3776,   711]))


---
## 5. Router A — ViT + Linear + Softmax (Task 2.5)
**Deep Learning baseline.** Capa lineal entrenada sobre los CLS tokens.

In [7]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

class LinearGatingHead(nn.Module):
    """Router A: CLS token → Linear → Softmax → expert_id"""
    def __init__(self, d_model=192, n_experts=5):
        super().__init__()
        self.gate = nn.Linear(d_model, n_experts)

    def forward(self, z):
        return F.softmax(self.gate(z), dim=-1)  # (B, 5)

# ─── Preparar datos como tensores ────────────────────────────────────────────
n_experts = len(np.unique(y_train))
print(f'Número de expertos detectados: {n_experts}')

X_tr = torch.from_numpy(Z_train)
X_va = torch.from_numpy(Z_val)
# Para el router, la etiqueta es el expert_id (qué experto debería activarse)
y_tr_t = torch.from_numpy(y_train.astype(np.int64))
y_va_t = torch.from_numpy(y_val.astype(np.int64))

train_ds_router = TensorDataset(X_tr, y_tr_t)
val_ds_router   = TensorDataset(X_va, y_va_t)
train_loader_r  = DataLoader(train_ds_router, batch_size=512, shuffle=True)
val_loader_r    = DataLoader(val_ds_router,   batch_size=512, shuffle=False)

# ─── Entrenar Router A ───────────────────────────────────────────────────────
router_a = LinearGatingHead(d_model=192, n_experts=n_experts).to(DEVICE)
opt_a    = torch.optim.Adam(router_a.parameters(), lr=1e-3)
crit_a   = nn.CrossEntropyLoss()

best_acc_a = 0.0
results_a  = {}

for epoch in range(1, 31):
    router_a.train()
    for X_batch, y_batch in train_loader_r:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        logits = router_a.gate(X_batch)
        loss   = crit_a(logits, y_batch)
        opt_a.zero_grad(); loss.backward(); opt_a.step()

    # Validación
    router_a.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader_r:
            probs = router_a(X_batch.to(DEVICE))
            all_preds.extend(probs.argmax(dim=1).cpu().numpy())
            all_true.extend(y_batch.numpy())

    acc = accuracy_score(all_true, all_preds)
    if acc > best_acc_a:
        best_acc_a = acc
        torch.save(router_a.state_dict(), CHECKPOINT_DIR / 'router_a_best.pth')

    if epoch % 5 == 0:
        print(f'  Epoch {epoch:02d}/30 | Routing Acc: {acc:.4f}')

# Medir latencia
t0 = time.time()
with torch.no_grad():
    _ = router_a(X_va[:1000].to(DEVICE))
lat_a = (time.time() - t0) / 1000 * 1000  # ms por muestra

results_a = {
    'router': 'A — ViT + Linear',
    'tipo': 'DL (gradiente)',
    'routing_acc': round(best_acc_a, 4),
    'latencia_ms': round(lat_a, 2),
    'parametros': sum(p.numel() for p in router_a.parameters()),
    'requiere_gpu': True,
}
print(f'\n✅ Router A — Mejor Routing Accuracy: {best_acc_a:.4f}')
print(f'   Latencia: {lat_a:.2f} ms/muestra')

Número de expertos detectados: 4
  Epoch 05/30 | Routing Acc: 0.9966
  Epoch 10/30 | Routing Acc: 0.9975
  Epoch 15/30 | Routing Acc: 0.9985
  Epoch 20/30 | Routing Acc: 0.9986
  Epoch 25/30 | Routing Acc: 0.9984
  Epoch 30/30 | Routing Acc: 0.9990

✅ Router A — Mejor Routing Accuracy: 0.9990
   Latencia: 0.00 ms/muestra


---
## 6. Router B — GMM (Task 2.6)
**Paramétrico estadístico.** Gaussian Mixture Model con 5 componentes, ajuste con EM.

In [8]:
from sklearn.mixture import GaussianMixture

print('Entrenando Router B (GMM)...')
t0 = time.time()

gmm = GaussianMixture(
    n_components=n_experts,
    covariance_type='diag',  # 'full' si tienes >5000 muestras por experto
    max_iter=200,
    random_state=SEED,
    verbose=1,
    verbose_interval=50,
)
gmm.fit(Z_train)
elapsed = time.time() - t0
print(f'  GMM ajustado en {elapsed:.1f}s')

# Validación
t_inf = time.time()
probs_b    = gmm.predict_proba(Z_val)   # (N_val, 5)
preds_b    = probs_b.argmax(axis=1)
lat_b      = (time.time() - t_inf) / len(Z_val) * 1000
acc_b      = accuracy_score(y_val, preds_b)

# Nota: GMM asigna componentes por similitud estadística, no por expert_id directo
# La routing accuracy real requiere mapear componentes → expertos
# Usamos el mapeo por mayoría de votos
from scipy.stats import mode
component_to_expert = {}
for comp in range(n_experts):
    mask = preds_b == comp
    if mask.sum() > 0:
        expert = mode(y_val[mask], keepdims=True).mode[0]
        component_to_expert[comp] = expert

preds_b_mapped = np.array([component_to_expert.get(p, p) for p in preds_b])
acc_b_mapped   = accuracy_score(y_val, preds_b_mapped)

results_b = {
    'router': 'B — GMM',
    'tipo': 'Paramétrico (EM)',
    'routing_acc': round(acc_b_mapped, 4),
    'latencia_ms': round(lat_b, 2),
    'parametros': f'{n_experts}×(d+d) = {n_experts * 2 * 192}',
    'requiere_gpu': False,
}
print(f'\n✅ Router B (GMM) — Routing Accuracy: {acc_b_mapped:.4f}')
print(f'   Latencia: {lat_b:.2f} ms/muestra')

Entrenando Router B (GMM)...
Initialization 0
Initialization converged.
  GMM ajustado en 3.5s

✅ Router B (GMM) — Routing Accuracy: 0.9015
   Latencia: 0.00 ms/muestra


---
## 7. Router C — Naive Bayes (Task 2.7)
**Paramétrico analítico.** Sin optimización iterativa — solución en forma cerrada.

In [9]:
from sklearn.naive_bayes import GaussianNB

print('Entrenando Router C (Naive Bayes)...')
t0 = time.time()

nb = GaussianNB()
nb.fit(Z_train, y_train)
elapsed = time.time() - t0
print(f'  Naive Bayes ajustado en {elapsed:.3f}s (analítico)')

# Validación
t_inf   = time.time()
preds_c = nb.predict(Z_val)
probs_c = nb.predict_proba(Z_val)
lat_c   = (time.time() - t_inf) / len(Z_val) * 1000
acc_c   = accuracy_score(y_val, preds_c)

results_c = {
    'router': 'C — Naive Bayes',
    'tipo': 'Paramétrico (MLE analítico)',
    'routing_acc': round(acc_c, 4),
    'latencia_ms': round(lat_c, 3),
    'parametros': f'{n_experts}×2d = {n_experts * 2 * 192}',
    'requiere_gpu': False,
}
print(f'\n✅ Router C (Naive Bayes) — Routing Accuracy: {acc_c:.4f}')
print(f'   Latencia: {lat_c:.3f} ms/muestra')

Entrenando Router C (Naive Bayes)...
  Naive Bayes ajustado en 0.227s (analítico)

✅ Router C (Naive Bayes) — Routing Accuracy: 0.9114
   Latencia: 0.004 ms/muestra


---
## 8. Router D — sklearn k-NN + PCA (Task 2.8)
**No paramétrico.** k-NN con métrica coseno usando `sklearn.neighbors.KNeighborsClassifier`.

> **Tip del profesor:** Aplicar PCA a d=32 antes del k-NN para evitar la maldición de la dimensionalidad con 192 dims.

In [10]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

# ─── PCA: 192 → 32 dims (tip del profesor) ───────────────────────────────────
print('Aplicando PCA (192 → 32 dims)...')
pca = PCA(n_components=32, random_state=SEED)
Z_train_pca = pca.fit_transform(Z_train).astype(np.float32)
Z_val_pca   = pca.transform(Z_val).astype(np.float32)
var_explained = pca.explained_variance_ratio_.sum()
print(f'  Varianza explicada con 32 componentes: {var_explained:.2%}')

# ─── sklearn k-NN con métrica coseno ─────────────────────────────────────────
K = 5  # número de vecinos

print('Entrenando Router D (sklearn k-NN coseno)...')
t0 = time.time()

knn = KNeighborsClassifier(
    n_neighbors=K,
    metric='cosine',
    algorithm='brute',   # necesario para métrica coseno
    n_jobs=-1,
)
knn.fit(Z_train_pca, y_train)
print(f'  Índice construido: {len(Z_train_pca)} vectores en {time.time()-t0:.1f}s')

# ─── Inferencia k-NN ─────────────────────────────────────────────────────────
t_inf  = time.time()
preds_d = knn.predict(Z_val_pca)
lat_d   = (time.time() - t_inf) / len(Z_val_pca) * 1000
acc_d   = accuracy_score(y_val, preds_d)

results_d = {
    'router': 'D — k-NN + PCA (sklearn)',
    'tipo': 'No paramétrico',
    'routing_acc': round(acc_d, 4),
    'latencia_ms': round(lat_d, 2),
    'parametros': f'N×32 = {len(Z_train_pca) * 32} (todos los train)',
    'requiere_gpu': False,
}
print(f'\n✅ Router D (k-NN sklearn) — Routing Accuracy: {acc_d:.4f}')
print(f'   Latencia: {lat_d:.2f} ms/muestra')
print(f'   Varianza PCA preservada: {var_explained:.2%}')

Aplicando PCA (192 → 32 dims)...
  Varianza explicada con 32 componentes: 84.53%
Entrenando Router D (sklearn k-NN coseno)...
  Índice construido: 121287 vectores en 0.0s

✅ Router D (k-NN sklearn) — Routing Accuracy: 0.9931
   Latencia: 1.72 ms/muestra
   Varianza PCA preservada: 84.53%


---
## 9. Ablation Study — Tabla Comparativa (Task 2.9)

In [11]:
import pandas as pd

# Compilar resultados
all_results = [results_a, results_b, results_c, results_d]

df_ablation = pd.DataFrame(all_results)[[
    'router', 'tipo', 'routing_acc', 'latencia_ms', 'requiere_gpu'
]]
df_ablation.columns = ['Router', 'Tipo', 'Routing Acc', 'Latencia (ms)', 'GPU']
df_ablation = df_ablation.sort_values('Routing Acc', ascending=False).reset_index(drop=True)

print('=' * 65)
print('ABLATION STUDY — Comparativa de Routers')
print('=' * 65)
print(df_ablation.to_string(index=False))
print('=' * 65)

# Router ganador
winner = df_ablation.iloc[0]
print(f'\n🏆 Router ganador: {winner["Router"]}')
print(f'   Routing Accuracy: {winner["Routing Acc"]}')
print(f'   Latencia:         {winner["Latencia (ms)"]} ms')

# Meta del proyecto
meta_acc = 0.80
best_acc = df_ablation['Routing Acc'].max()
print(f'\n🎯 Meta Routing Acc > {meta_acc}: {"✅" if best_acc >= meta_acc else "⚠️"} ({best_acc:.4f})')

# Guardar tabla para el reporte técnico
df_ablation.to_csv(EMBEDDINGS_DIR / 'ablation_study.csv', index=False)
print(f'\n📄 Tabla guardada → {EMBEDDINGS_DIR}/ablation_study.csv')

ABLATION STUDY — Comparativa de Routers
                  Router                        Tipo  Routing Acc  Latencia (ms)   GPU
        A — ViT + Linear              DL (gradiente)       0.9990          0.000  True
D — k-NN + PCA (sklearn)              No paramétrico       0.9931          1.720 False
         C — Naive Bayes Paramétrico (MLE analítico)       0.9114          0.004 False
                 B — GMM            Paramétrico (EM)       0.9015          0.000 False

🏆 Router ganador: A — ViT + Linear
   Routing Accuracy: 0.999
   Latencia:         0.0 ms

🎯 Meta Routing Acc > 0.8: ✅ (0.9990)

📄 Tabla guardada → /workspace/moe_medical_vision/embeddings/ablation_study.csv


## 10. Resumen final

In [12]:
print('=' * 55)
print('RESUMEN EPIC 2')
print('=' * 55)

tasks = [
    ('2.4', 'Extracción CLS tokens',   '✅'),
    ('2.5', 'Router A (ViT+Linear)',    '✅'),
    ('2.6', 'Router B (GMM)',           '✅'),
    ('2.7', 'Router C (Naive Bayes)',   '✅'),
    ('2.8', 'Router D (FAISS k-NN)',    '✅'),
    ('2.9', 'Tabla Ablation Study',     '✅'),
]
for tid, name, status in tasks:
    print(f'  {status} Task {tid}: {name}')

winner_name = df_ablation.iloc[0]['Router']
winner_acc  = df_ablation.iloc[0]['Routing Acc']

print(f'\n🏆 Router ganador para MoE_System: {winner_name}')
print(f'   Routing Accuracy: {winner_acc}')
print()
print('Siguiente → 03_moe_system.ipynb')
print('  Task 3.1: Unir Backbone + Router ganador + 5 Expertos')
print('  Task 3.2: Auxiliary Loss (Switch Transformer)')
print('  Task 3.3: Fine-Tuning global')

RESUMEN EPIC 2
  ✅ Task 2.4: Extracción CLS tokens
  ✅ Task 2.5: Router A (ViT+Linear)
  ✅ Task 2.6: Router B (GMM)
  ✅ Task 2.7: Router C (Naive Bayes)
  ✅ Task 2.8: Router D (FAISS k-NN)
  ✅ Task 2.9: Tabla Ablation Study

🏆 Router ganador para MoE_System: A — ViT + Linear
   Routing Accuracy: 0.999

Siguiente → 03_moe_system.ipynb
  Task 3.1: Unir Backbone + Router ganador + 5 Expertos
  Task 3.2: Auxiliary Loss (Switch Transformer)
  Task 3.3: Fine-Tuning global


In [13]:
import torch, json
from pathlib import Path

CHECKPOINT_DIR = Path('/workspace/moe_medical_vision/checkpoints')
METRICS_DIR    = CHECKPOINT_DIR / 'metrics'

# ── Experto 1 (NIH) — ensemble v9 de 6 binarios ──────────────────────────────
meta_files = sorted(METRICS_DIR.glob('expert1_ensemble_meta_*.json'))
if meta_files:
    with open(meta_files[-1]) as f:
        meta1 = json.load(f)
    f1_1  = meta1['ensemble_macro_f1_retuned_thr']
    auc_1 = meta1['ensemble_macro_auc']
    n_ck  = sum(len(s['ckpt']) if isinstance(s['ckpt'], list) else 1
                for s in meta1['specialists'])
    print(f'Experto 1 NIH (ensemble v9): F1={f1_1:.4f}, AUC={auc_1:.4f} '
          f'({n_ck} ckpts en specialists/)')
    for sp in meta1['specialists']:
        tag = f'{len(sp["ckpt"])}-seed' if isinstance(sp['ckpt'], list) else 'single'
        print(f'  - {sp["name"]:<14s} F1={sp["f1"]:.4f}  thr={sp["thr"]:.2f}  ({tag})')
else:
    print('No se encontró ensemble metadata — corre primero notebook 01')

# ── Expertos 2-5 (single-model) ──────────────────────────────────────────────
otros = {
    'Experto 2 ISIC':       'expert2_isic_best.pth',
    'Experto 3 OA':         'expert3_oa_best.pth',
    'Experto 4 LUNA16':     'expert4_luna16_MIP_best.pth',
    'Experto 5 Pancreatic': 'expert5_pancreatic_FAST_best.pth',
}
for nombre, archivo in otros.items():
    p = CHECKPOINT_DIR / archivo
    if not p.exists():
        print(f'{nombre}: (no checkpoint)'); continue
    ck = torch.load(p, map_location='cpu', weights_only=False)
    f1 = ck.get('best_val_f1', ck.get('best_f1', '?'))
    print(f'{nombre}: F1={f1}')


Experto 1 NIH (ensemble v9): F1=0.5484, AUC=0.7494 (10 ckpts en specialists/)
  - Global         F1=0.6577  thr=0.47  (single)
  - Atelectasis    F1=0.5431  thr=0.63  (single)
  - Effusion       F1=0.6777  thr=0.66  (single)
  - Infiltration   F1=0.6399  thr=0.40  (single)
  - Mass           F1=0.4688  thr=0.76  (3-seed)
  - Nodule         F1=0.3912  thr=0.67  (3-seed)
Experto 2 ISIC: F1=0.7920855935764763
Experto 3 OA: F1=0.8356692771952554
Experto 4 LUNA16: F1=0.6153187565858799
Experto 5 Pancreatic: F1=0.7552910052910053


In [14]:
import pandas as pd
df = pd.read_csv('/workspace/moe_medical_vision/embeddings/ablation_study.csv')
print(df.to_string())

                     Router                         Tipo  Routing Acc  Latencia (ms)    GPU
0          A — ViT + Linear               DL (gradiente)       0.9990          0.000   True
1  D — k-NN + PCA (sklearn)               No paramétrico       0.9931          1.720  False
2           C — Naive Bayes  Paramétrico (MLE analítico)       0.9114          0.004  False
3                   B — GMM             Paramétrico (EM)       0.9015          0.000  False
